# Province reporting-area Analytics validation

Validate the five D-55 views against current Gold. Evidence is merged into `analytics_dq_results` before the STOP gate.


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;
DECLARE OR REPLACE VARIABLE province_analytics_rule STRING
    DEFAULT 'province-reporting-analytics-validation-v1';

CREATE OR REPLACE TEMPORARY VIEW province_analytics_validation_context AS
SELECT SHA2(CONCAT_WS('||', MAX(run_id), MAX(source_snapshot_id),
        province_analytics_rule), 256) AS validation_run_id,
    MAX(run_id) AS pipeline_run_id, MAX(source_snapshot_id) AS source_snapshot_id
FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE;


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW province_analytics_checks AS
WITH checks AS (
    SELECT 'analytics_reporting_area_investment' AS table_name, 'Uniqueness' AS attribute,
        'one row per reporting area and current snapshot' AS check_name,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT('area', reporting_area_key,
            'snapshot', source_snapshot_id)) AS failed_rows, COUNT(*) AS total_rows, 'stop' AS action
    FROM `05-analytics`.analytics_reporting_area_investment
    UNION ALL
    SELECT 'analytics_reporting_area_investment', 'Consistency',
        'mapped plus Unknown project totals equal current Gold',
        ABS(SUM(project_count) - (SELECT COUNT(*) FROM `03-gold`.fact_project_snapshot
            WHERE is_current_snapshot = TRUE)), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_investment
    UNION ALL
    SELECT 'analytics_reporting_area_investment', 'Consistency',
        'mapped plus Unknown budget equals current Gold',
        CASE WHEN SUM(total_reported_budget_pesos) <=> (
            SELECT SUM(reported_budget_pesos) FROM `03-gold`.fact_project_snapshot
            WHERE is_current_snapshot = TRUE
        ) THEN 0 ELSE 1 END, COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_investment
    UNION ALL
    SELECT 'analytics_reporting_area_investment', 'Validity',
        'budget share uses the national mapped plus Unknown denominator',
        CASE WHEN SUM(total_reported_budget_pesos) IS NULL THEN 0
            WHEN ABS(SUM(national_budget_share_pct) - 100.0) <= 0.01 THEN 0 ELSE 1 END,
        COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_investment
    UNION ALL
    SELECT 'analytics_reporting_area_investment', 'Referential integrity',
        'every reporting area key exists in the dimension',
        COUNT_IF(area.reporting_area_key IS NULL), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_investment AS view
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON view.reporting_area_key = area.reporting_area_key
    UNION ALL
    SELECT 'analytics_reporting_area_category', 'Uniqueness',
        'category view grain is unique', COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'area', reporting_area_key, 'sector', standardized_sector,
            'category_state', infra_type_mapping_status,
            'area_state', reporting_area_match_status, 'snapshot', source_snapshot_id)),
        COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_category
    UNION ALL
    SELECT 'analytics_reporting_area_category', 'Consistency',
        'category project counts reconcile to current Gold',
        ABS(SUM(project_count) - (SELECT COUNT(*) FROM `03-gold`.fact_project_snapshot
            WHERE is_current_snapshot = TRUE)), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_category
    UNION ALL
    SELECT 'analytics_reporting_area_category', 'Consistency',
        'category budget reconciles to current Gold',
        CASE WHEN SUM(total_reported_budget_pesos) <=> (
            SELECT SUM(reported_budget_pesos) FROM `03-gold`.fact_project_snapshot
            WHERE is_current_snapshot = TRUE
        ) THEN 0 ELSE 1 END, COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_category
    UNION ALL
    SELECT 'analytics_reporting_area_delivery_status', 'Uniqueness',
        'delivery view grain is unique', COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'area', reporting_area_key, 'sector', standardized_sector,
            'category_state', infra_type_mapping_status, 'status', status_group,
            'status_state', status_mapping_status, 'snapshot', source_snapshot_id)),
        COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_delivery_status
    UNION ALL
    SELECT 'analytics_reporting_area_delivery_status', 'Consistency',
        'delivery project counts reconcile to current Gold',
        ABS(SUM(project_count) - (SELECT COUNT(*) FROM `03-gold`.fact_project_snapshot
            WHERE is_current_snapshot = TRUE)), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_delivery_status
    UNION ALL
    SELECT 'analytics_reporting_area_population_equity', 'Uniqueness',
        'population equity has one row per comparable area, snapshot, and year',
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT('area', reporting_area_key,
            'snapshot', source_snapshot_id, 'year', reference_year)), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_population_equity
    UNION ALL
    SELECT 'analytics_reporting_area_population_equity', 'Validity',
        'investment and population shares use the same comparable set',
        CASE WHEN COUNT(*) = 0 THEN 1
            WHEN ABS(SUM(investment_share_pct) - 100.0) <= 0.01
             AND ABS(SUM(comparable_population_share_pct) - 100.0) <= 0.01
            THEN 0 ELSE 1 END, COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_population_equity
    UNION ALL
    SELECT 'analytics_reporting_area_population_equity', 'Completeness',
        'comparable population and investment values are available',
        COUNT_IF(population_count IS NULL OR population_count <= 0
            OR total_reported_budget_pesos IS NULL), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_population_equity
    UNION ALL
    SELECT 'analytics_reporting_area_project_size_concentration', 'Uniqueness',
        'project size view grain is unique',
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT('area', reporting_area_key,
            'snapshot', source_snapshot_id)), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_project_size_concentration
    UNION ALL
    SELECT 'analytics_reporting_area_project_size_concentration', 'Consistency',
        'project size totals reconcile to current Gold',
        ABS(SUM(project_count) - (SELECT COUNT(*) FROM `03-gold`.fact_project_snapshot
            WHERE is_current_snapshot = TRUE)), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_project_size_concentration
    UNION ALL
    SELECT 'analytics_reporting_area_project_size_concentration', 'Completeness',
        'unavailable budgets remain NULL rather than zero',
        COUNT_IF(projects_with_budget = 0 AND total_reported_budget_pesos IS NOT NULL),
        COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_project_size_concentration
    UNION ALL
    SELECT 'analytics_reporting_area_investment', 'Consistency', 'one current DPWH snapshot',
        GREATEST(COUNT(DISTINCT source_snapshot_id) - 1, 0), COUNT(*), 'stop'
    FROM `05-analytics`.analytics_reporting_area_investment
)
SELECT context.validation_run_id, context.pipeline_run_id, 'analytics' AS layer,
    checks.table_name, context.source_snapshot_id, province_analytics_rule AS rule_version,
    checks.attribute AS data_quality_attribute, checks.check_name AS data_quality_check,
    CAST(checks.failed_rows AS BIGINT) AS failed_rows, CAST(checks.total_rows AS BIGINT) AS total_rows,
    ROUND(100.0 * checks.failed_rows / NULLIF(checks.total_rows, 0), 4) AS percentage,
    CASE WHEN checks.failed_rows > 0 THEN 'FAIL' ELSE 'PASS' END AS status, checks.action,
    CONCAT('failed=', checks.failed_rows, '; total=', checks.total_rows) AS details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks CROSS JOIN province_analytics_validation_context AS context;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.analytics_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
) USING DELTA COMMENT 'Analytics validation evidence. One row per check per validation run.';

MERGE INTO `04-validation`.analytics_dq_results AS target
USING province_analytics_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking province-reporting Analytics checks failed')
)
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM province_analytics_validation_context);


# Summary

The validator checks all five D-55 grains, one current snapshot, reporting-area keys, mapped plus Unknown reconciliation, project and budget totals, comparable population denominators, and NULL preservation. Databricks runtime evidence remains pending.
